# Week 36, Day 2: Fisher information: how well could any decoder do?

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w036/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The maximum-likelihood decoder and the bound

Write fisher(N), returning T times the sum over neurons of f'(θ0)² / f(θ0) for a population of N von Mises neurons built by pop(N). The cell then compares the Cramér-Rao standard deviation with the spread of maximum-likelihood estimates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1999)
T, th0 = 0.2, 0.3

def pop(N, kappa=2.0, rmax=20.0, b=1.0):
    pref = np.linspace(-np.pi, np.pi, N, endpoint=False)
    f = lambda th: b + rmax * np.exp(kappa * (np.cos(th - pref) - 1))
    fp = lambda th: -rmax * kappa * np.sin(th - pref) * np.exp(kappa * (np.cos(th - pref) - 1))
    return f, fp

def fisher(N):
    f, fp = pop(N)
    return ...

grid = np.linspace(-np.pi, np.pi, 2000, endpoint=False)
cr, ml_sd = {}, {}
for N in (10, 20, 50, 100):
    f, fp = pop(N)
    F = np.array([f(g) for g in grid]) * T; lF = np.log(F)
    est = np.array([grid[np.argmax(lF @ r.poisson(f(th0) * T) - F.sum(1))] for _ in range(2000)])
    cr[N] = np.degrees(1 / np.sqrt(fisher(N)))
    ml_sd[N] = np.degrees(np.angle(np.exp(1j * (est - th0))).std())
    print(f"N = {N:3d}: bound {cr[N]:5.2f} deg, ML decoder {ml_sd[N]:5.2f} deg")

In [ ]:
import numpy as np
def test_formula():
    "fisher(N) matches a numerical derivative of the tuning curves"
    f, fp = pop(30); e = 1e-5
    num = T * np.sum(((f(th0 + e) - f(th0 - e)) / (2 * e)) ** 2 / f(th0))
    assert abs(fisher(30) - num) / num < 1e-4

def test_bound():
    "for 20 or more neurons, the ML decoder's standard deviation is within 10% of the bound"
    for N in (20, 50, 100):
        assert 0.95 * cr[N] < ml_sd[N] < 1.1 * cr[N]

test_formula()
test_bound()
print("All checks passed")

### Exercise: Information lives on the flanks

For a population of 72 neurons, compute each neuron's contribution T f'(θ0)² / f(θ0) and the offset of its preferred direction from θ0 in degrees; find the offset of the most informative neuron.

In [ ]:
f, fp = pop(72)
pref = np.linspace(-np.pi, np.pi, 72, endpoint=False)
contrib = ...
offsets = np.degrees(np.angle(np.exp(1j * (pref - th0))))
best_offset = offsets[np.argmax(contrib)]
print(f"most informative neuron prefers {best_offset:+.0f} deg from the stimulus")
o = np.argsort(offsets)
plt.plot(offsets[o], contrib[o], ".-"); plt.xlabel("preferred minus stimulus (deg)"); plt.ylabel("Fisher information contribution"); plt.show()

In [ ]:
import numpy as np
def test_flank():
    "the most informative neuron prefers an orientation between 20 and 80 degrees from the stimulus"
    assert 20 < abs(best_offset) < 80

def test_peak_low():
    "the neuron preferring the stimulus itself contributes less than 5% of the maximum"
    assert contrib[np.argmin(np.abs(offsets))] < 0.05 * contrib.max()

test_flank()
test_peak_low()
print("All checks passed")

### Exercise: Information as the population grows

Build the four covariance matrices for each population size and compute the linear Fisher information fp @ inv(C) @ fp for each (use np.linalg.solve). Counts are in a 0.2 s window, so the mean is f T, the slope f' T and the independent variance f T.

In [ ]:
eps = 1e-3
J = {}
for N in (10, 50, 100, 200, 500, 1000):
    f, fp = pop(N)
    pref = np.linspace(-np.pi, np.pi, N, endpoint=False)
    d, var = fp(th0) * T, f(th0) * T
    sd = np.sqrt(var)
    C = {"independent": np.diag(var)}
    U = 0.1 * np.outer(sd, sd); np.fill_diagonal(U, var); C["uniform"] = U
    rho = 0.2 * np.exp(-np.abs(np.angle(np.exp(1j * (pref[:, None] - pref[None, :])))) / 0.5)
    Lr = rho * np.outer(sd, sd); np.fill_diagonal(Lr, var); C["limited"] = Lr
    C["differential"] = np.diag(var) + eps * np.outer(d, d)
    J[N] = {k: ... for k, M in C.items()}
    print(N, {k: round(float(v), 1) for k, v in J[N].items()})
for k in J[10]:
    plt.loglog(list(J), [J[N][k] for N in J], "o-", label=k)
plt.axhline(1 / eps, color="gray", ls=":"); plt.xlabel("neurons"); plt.ylabel("linear Fisher information"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_uniform():
    "uniform correlations do not reduce information"
    assert J[1000]["uniform"] >= 0.95 * J[1000]["independent"]

def test_limits():
    "limited-range and differential correlations make information saturate"
    assert J[1000]["limited"] < 0.1 * J[1000]["independent"]
    assert J[1000]["differential"] < 1 / eps and J[1000]["differential"] < 1.5 * J[500]["differential"]

test_uniform()
test_limits()
print("All checks passed")